# Workflows and Deadlines

Level 5: Senior FDE / Staff-Level Scenarios.

**Problems in this notebook**

- Problem 44: Multi-Step Workflow State
- Problem 49: SLA Breach Detection

**How to use this notebook**

1. Read the problem and say your plan out loud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `04_workflows_and_deadlines_THEORY.md`.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 44: Multi-Step Workflow State

**The situation**

An onboarding agent collects five things over several chat turns: company name, use case, data source, security approval and deployment region. The answers live in a saved state dict, not in the prompt. A session idle for more than a day should start over.

**Your task**

Write `next_workflow_step(state, required_steps, now, max_idle_seconds=86400)` that returns the first step still missing, `"expired"` if the session sat idle too long, or `None` when everything is done.

**Example**

`state={"company": "Acme", "updated_at": 0}`, `required=["company", "use_case"]`, `now=100` -> `"use_case"`

**Ask before coding**

- Do the steps have to happen in a fixed order?
- Can a user go back and change an earlier answer?
- When does an unfinished workflow expire?

**Plan to say aloud**

1. If updated_at is set and too old, return "expired".
2. Walk the required steps in order.
3. A step is missing if it is absent, None, False or a blank string.
4. Return the first missing step, or None when all are filled.

### Prompt for your AI assistant

```text
Write a Python function next_workflow_step(state: dict, required_steps: list[str], now: int,
max_idle_seconds: int = 86400) -> str | None.
Rules: if state has "updated_at" and now - updated_at > max_idle_seconds, return "expired".
Otherwise return the first step in required_steps whose value is missing, None, False, or a
blank/whitespace string. Return None if all are filled. A missing updated_at means a new workflow.
Do not call time.time(). Show only the function.
```

**Check the AI's code for**

- Uses `step in state`, so `"use_case": "   "` or `"security_approval": False` counts as done.
- Checks the steps before the expiry, so a stale session keeps going.
- Returns steps in dict order instead of the required order.
- Reads the real clock instead of the `now` argument.

In [ ]:
def next_workflow_step(state: dict, required_steps: list[str], now: int,
                       max_idle_seconds: int = 86400) -> str | None:
    """Return the next missing step, "expired", or None when done.

    Rules:
    - updated_at present and now - updated_at > max_idle_seconds -> "expired"
    - a step is missing if absent, None, False or a blank string
    - return the first missing step in required_steps order, else None
    """
    raise NotImplementedError

In [ ]:
STEPS = ["company", "use_case", "data_source", "security_approval", "region"]
full = {"company": "Acme", "use_case": "support", "data_source": "SharePoint",
        "security_approval": False, "region": "eu", "updated_at": 0}
cases = [
    (({"company": "Acme", "updated_at": 0}, ["company", "use_case"], 100), "use_case"),
    (({"company": "Acme", "use_case": "support", "updated_at": 0}, ["company", "use_case"], 100), None),
    (({}, STEPS, 0), "company"),                                                   # brand new
    (({"company": "Acme", "use_case": "   ", "updated_at": 0}, ["company", "use_case"], 10), "use_case"),
    ((full, STEPS, 10), "security_approval"),                                      # False is not approved
    (({"company": "Acme", "updated_at": 0}, ["company", "use_case"], 90000), "expired"),
]
run_checks(next_workflow_step, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def next_workflow_step_reference(state: dict, required_steps: list[str], now: int,
                                 max_idle_seconds: int = 86400) -> str | None:
    updated_at = state.get("updated_at")
    if updated_at is not None and now - updated_at > max_idle_seconds:
        return "expired"
    for step in required_steps:
        value = state.get(step)
        missing = value is None or value is False or (isinstance(value, str) and not value.strip())
        if missing:
            return step
    return None


assert run_checks(next_workflow_step_reference, cases)

**Explain it in 30 seconds**

> "The prompt history is not the workflow state. I keep the answers in a saved dict and check it each turn. First I check if the session expired, then I return the first required step that is still empty. Blank text and a False approval both count as missing. The time is passed in, so the expiry is easy to test."

**Edge cases to mention**

- A brand-new session with no state at all.
- A value of `0` or `False` that might be a real answer.
- A user who changes an earlier answer, like switching the region.
- A session exactly at the idle limit.

**If they ask "how would this work in production?"**

- Save the state in a database as a state machine, with allowed moves from step to step.
- Validate each answer when it is given, for example a region from an allowed list.
- Keep a history of changes, so an audit can see who approved security and when.

## Problem 49: SLA Breach Detection

**The situation**

An enterprise contract promises that high-priority tickets get an AI triage reply within five minutes. SLA (service level agreement) reports feed into credits and renewals, so they must be right. Tickets with no reply yet still count: they may already be late.

**Your task**

Write `detect_sla_breaches(tickets, sla_seconds, now)` that returns the ids of tickets that took longer than their priority's limit, in input order. `now` is passed in.

**Example**

Ticket `created=0, responded=400, priority="high"`, `sla_seconds={"high": 300}`, `now=500` -> `["t1"]`

**Ask before coding**

- Are timestamps in the same time zone and unit?
- Does the limit change by priority?
- What about tickets with no reply yet?

**Plan to say aloud**

1. Default a missing priority to "normal".
2. Skip tickets with no created time or a priority not in the SLA map.
3. Use the reply time, or now if there is no reply.
4. It is a breach if elapsed time is more than the limit.

### Prompt for your AI assistant

```text
Write a Python function detect_sla_breaches(tickets: list[dict], sla_seconds: dict[str, int],
now: int) -> list[str].
Rules: priority defaults to "normal". Skip tickets with no "created" or whose priority is not in
sla_seconds. End time is ticket["responded"] if it is not None, else now.
A breach is end - created > limit (exactly at the limit is fine). Keep input order.
Do not call time.time(). Show only the function.
```

**Check the AI's code for**

- Skips tickets with no reply instead of using `now`, which hides the worst breaches.
- Uses `ticket.get("responded") or now`, which treats a reply at time 0 as no reply.
- Uses `>=`, so a reply exactly at the limit counts as a breach.
- Crashes on a priority missing from the SLA map.

In [ ]:
def detect_sla_breaches(tickets: list[dict], sla_seconds: dict[str, int], now: int) -> list[str]:
    """Return ids of tickets that broke their SLA, in input order.

    Rules:
    - priority defaults to "normal"
    - skip tickets with no created time or a priority not in sla_seconds
    - end = responded if not None, else now
    - breach if end - created > limit
    """
    raise NotImplementedError

In [ ]:
SLA = {"high": 300, "normal": 3600}
cases = [
    (([{"id": "t1", "priority": "high", "created": 0, "responded": 400}], SLA, 500), ["t1"]),
    (([{"id": "t2", "priority": "high", "created": 0, "responded": 300}], SLA, 500), []),   # exactly at limit
    (([{"id": "t3", "priority": "high", "created": 0}], SLA, 500), ["t3"]),               # no reply yet, already late
    (([{"id": "t4", "priority": "high", "created": 0}], SLA, 200), []),                   # no reply, still in time
    (([{"id": "t5", "created": 0, "responded": 4000},                                      # missing priority -> normal
       {"id": "t6", "priority": "p9", "created": 0, "responded": 9999},                    # unknown priority: skip
       {"id": "t7", "priority": "high", "responded": 9999}], SLA, 10000), ["t5"]),         # no created: skip
]
run_checks(detect_sla_breaches, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def detect_sla_breaches_reference(tickets: list[dict], sla_seconds: dict[str, int], now: int) -> list[str]:
    breaches: list[str] = []
    for ticket in tickets:
        limit = sla_seconds.get(ticket.get("priority", "normal"))
        created = ticket.get("created")
        if limit is None or created is None:
            continue
        end = ticket.get("responded")
        if end is None:
            end = now
        if end - created > limit:
            breaches.append(ticket["id"])
    return breaches


assert run_checks(detect_sla_breaches_reference, cases)

**Explain it in 30 seconds**

> "For each ticket I look up the limit for its priority and measure how long it waited. If there is no reply yet, I measure up to now, because an open ticket can already be late. Anything over the limit is a breach. The current time is passed in, so the report is repeatable and easy to test."

**Edge cases to mention**

- A ticket with no reply yet.
- A ticket with a priority the contract does not mention.
- Business hours: a ticket created Friday night may not count until Monday.
- Paused tickets, for example while waiting on the customer.

**If they ask "how would this work in production?"**

- Store time-zone-aware timestamps and use each customer's business calendar.
- Alert before the breach, for example at 80 percent of the limit, so someone can still act.
- Keep the SLA rules per contract, since different customers pay for different limits.

## Wrap-up

- Long-running work needs saved state, not memory inside a prompt.
- Time is an input. Pass `now` in so expiry and deadline rules are repeatable.
- Unfinished work still counts: an open ticket can be late, an idle session can expire.